# Top n de documentos recuperados por modelo (consulta de prueba)

Dado un **número de consulta** de la partición de prueba de MessIRve, este cuaderno devuelve los **n documentos mejor posicionados** (10 por omisión) de cada **modelo de recuperación inicial** de los experimentos, en el orden en que ese modelo los rankea.

Los rankings se leen de los *runs* que ya calculó el pipeline de recuperación (`~/.cache/messirve_embeddings/<modelo>/<configuración>/retrieval_run.lz4`): el cuaderno **no vuelve a recuperar** nada, no codifica embeddings ni usa GPU. Se ejecuta con el entorno `proyecto` (conda), que es el que tiene `ranx`, `datasets` y `pandas`.

**Modelos incluidos** (seis, ver §2): BM25, SPLADE-v3, multilingual-e5-large-instruct, BGE-M3, Qwen3-Embedding-0.6B y jina-embeddings-v5-text-small-retrieval. `microsoft/harrier-oss-v1-0.6b` **queda excluido** a propósito.

**Uso.** Fija `NUMERO_CONSULTA` en la §1 (es el `id` numérico de la consulta en el split de prueba; el ejemplo es `8101866`) y ejecuta el cuaderno completo. Para inspeccionar **otras** consultas está la **«Versión interactiva»** del final, con `analizar_consulta(numero, n)`, que recupera, carga los textos y muestra las tablas de la consulta que se le pase sin volver a ejecutar el resto; para *encontrar* un número están `listar_consultas()` y `buscar_consultas()` en la §3.

## 1. Configuración

- `NUMERO_CONSULTA`: número (`id`) de la consulta de prueba que se quiere inspeccionar.
- `N`: cuántos documentos devuelve cada modelo.
- `INCLUIR_TEXTO`: si es `True`, la §5 lee el corpus de párrafos de la Wikipedia en español para añadir el título y un extracto de cada documento; si es `False`, las tablas salen solo con `docid` y puntaje.

Los tres valores son los de la consulta de ejemplo; la «Versión interactiva» del final analiza otras consultas con `analizar_consulta(numero, n)`.

In [1]:
import gc
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------
NUMERO_CONSULTA = 8101866   # número (id) de la consulta de prueba a inspeccionar; ver §3
N = 10                      # documentos que devuelve cada modelo
INCLUIR_TEXTO = True        # True: título y extracto de cada documento (lee el corpus, §5)


# ---------------------------------------------------------------------------
# Entorno
# ---------------------------------------------------------------------------
def _raiz_repositorio() -> Path:
    """Carpeta de ir-spanish/ (la que contiene utils/), para importar el código del pipeline.

    El cuaderno vive en analysis/, pero JupyterLab lo abre con ese directorio como cwd y
    nbconvert lo abre donde se lance, así que la raíz se busca hacia arriba en vez de suponerla.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "utils" / "cache.py").exists():
            return base
    raise RuntimeError(
        "No se encontró la raíz de ir-spanish (la carpeta que contiene utils/). "
        "Abre el cuaderno desde el repositorio o ejecútalo desde su raíz."
    )


RAIZ = _raiz_repositorio()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from ranx.io import load_lz4          # lee un run tal como lo escribió ranx.Run.save
from utils import cache, data         # rutas de la caché y constantes del conjunto de datos

# `utils` deja el registro en nivel INFO y huggingface_hub registra cada petición HTTP, lo que
# llena de ruido la salida al leer el corpus (§5). Se silencian solo esos registros de terceros.
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

CACHE_DIR = Path.home() / ".cache" / "messirve_embeddings"

print(f"Repositorio:   {RAIZ}")
print(f"Caché de runs: {CACHE_DIR}")
print(f"Consulta:      {NUMERO_CONSULTA}   |   n = {N}   |   texto de los documentos: {INCLUIR_TEXTO}")

/home/jmendoza/miniconda3/envs/proyecto/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repositorio:   /home/jmendoza/ir-spanish
Caché de runs: /home/jmendoza/.cache/messirve_embeddings
Consulta:      8101866   |   n = 10   |   texto de los documentos: True


## 2. Modelos de recuperación inicial

Cada modelo tiene su *run* cacheado. La ruta se deriva con `utils.cache` —la misma que usa el pipeline— a partir de la configuración con la que se generó el run (`max_query_length` y `max_doc_length` de la tabla de resultados). BM25 no tiene longitudes de secuencia, así que `baselines/bm25.py` guarda su run en el directorio que solo depende del conjunto de datos; el mismo archivo está también en la ruta con longitudes (idénticos byte a byte), y se acepta cualquiera de las dos.

`microsoft/harrier-oss-v1-0.6b` **no se incluye** (petición del usuario, 2026-09-25). Queda en el registro como línea comentada, para que la exclusión sea explícita y no un olvido.

In [2]:
@dataclass(frozen=True)
class ModeloInicial:
    """Un modelo de recuperación inicial y la configuración con la que se cacheó su run."""
    nombre: str         # nombre en HuggingFace o identificador del pipeline
    alias: str          # nombre corto para las tablas
    max_q: int | None   # max_query_length del run (None: el modelo no trunca por longitudes)
    max_d: int | None   # max_doc_length del run


MODELOS = [
    # Léxico
    ModeloInicial("bm25_pyserini", "bm25", None, None),
    # Disperso (learned sparse)
    ModeloInicial("naver/splade-v3", "splade-v3", 512, 512),
    # Densos (dual-encoders)
    ModeloInicial("intfloat/multilingual-e5-large-instruct", "e5-large", 512, 512),
    ModeloInicial("BAAI/bge-m3", "bge-m3", 8192, 8192),
    ModeloInicial("Qwen/Qwen3-Embedding-0.6B", "qwen3-0.6b", 32768, 32768),
    ModeloInicial("jinaai/jina-embeddings-v5-text-small-retrieval", "jina-v5-small", 32768, 32768),
    # Excluido a propósito del análisis (2026-09-25):
    # ModeloInicial("microsoft/harrier-oss-v1-0.6b", "harrier", 32768, 32768),
]


def rutas_run(modelo: ModeloInicial) -> list[Path]:
    """Rutas donde puede estar el run de un modelo, en orden de preferencia.

    BM25 no tiene longitudes de secuencia: `baselines/bm25.py` guarda su run en el directorio
    que solo depende del conjunto de datos, y el mismo archivo se copió a la ruta con longitudes,
    que es la que usan los scripts de fusión. Se acepta cualquiera de las dos.
    """
    if modelo.max_q is None:
        solo_conjunto = CACHE_DIR / cache.model_slug(modelo.nombre) / (
            f"{data.COUNTRY}_v{data.DATASET_VERSION}_{cache._filter_suffix(data.MAX_WORD_COUNT)}"
        )
        con_longitudes = cache.cache_base(
            CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
            512, 512, data.MAX_WORD_COUNT,
        )
        return [cache.run_cache_path(solo_conjunto), cache.run_cache_path(con_longitudes)]

    base = cache.cache_base(
        CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
        modelo.max_q, modelo.max_d, data.MAX_WORD_COUNT,
    )
    return [cache.run_cache_path(base)]


def ruta_run(modelo: ModeloInicial) -> Path:
    """Primera ruta existente del run; falla con un mensaje claro si no hay ninguna."""
    candidatas = rutas_run(modelo)
    for ruta in candidatas:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No hay run cacheado para {modelo.nombre}. Rutas probadas:\n  "
        + "\n  ".join(str(ruta) for ruta in candidatas)
    )


# Comprobación: de qué archivo se va a leer cada modelo
filas = []
for modelo in MODELOS:
    run = ruta_run(modelo)
    filas.append({
        "alias": modelo.alias,
        "modelo": modelo.nombre,
        "max_query_length, max_doc_length": "—" if modelo.max_q is None else f"{modelo.max_q}, {modelo.max_d}",
        "run": str(run.relative_to(CACHE_DIR)),
        "MB": round(run.stat().st_size / 1024**2, 1),
    })

display(pd.DataFrame(filas))

,alias,modelo,"max_query_length, max_doc_length",run,MB
0,bm25,bm25_pyserini,—,bm25_pyserini/full_v1.2_nofilter/retrieval_run...,144.1
1,splade-v3,naver/splade-v3,"512, 512",naver__splade-v3/full_v1.2_q512_d512_nofilter/...,159.1
2,e5-large,intfloat/multilingual-e5-large-instruct,"512, 512",intfloat__multilingual-e5-large-instruct/full_...,148.1
3,bge-m3,BAAI/bge-m3,"8192, 8192",BAAI__bge-m3/full_v1.2_q8192_d8192_nofilter/re...,154.2
4,qwen3-0.6b,Qwen/Qwen3-Embedding-0.6B,"32768, 32768",Qwen__Qwen3-Embedding-0.6B/full_v1.2_q32768_d3...,154.5
5,jina-v5-small,jinaai/jina-embeddings-v5-text-small-retrieval,"32768, 32768",jinaai__jina-embeddings-v5-text-small-retrieva...,154.1


## 3. La consulta de prueba y su verdad de referencia

El **número de consulta** es el campo `id` del split de prueba (un entero, p. ej. `8101866`); las claves de los runs y de la verdad de referencia son ese número en forma de cadena. La partición de prueba tiene 170,055 consultas únicas.

La verdad de referencia se lee de `pruned_qrels.json`, el mismo archivo con el que el pipeline evalúa: para cada consulta, los `docid` relevantes. Un documento es un párrafo de un artículo de Wikipedia, identificado por un `docid` de la forma `"articulo#parrafo"` (p. ej. `"328242#0"`).

Para elegir un número: `listar_consultas(inicio, cuantas)` recorre la partición y `buscar_consultas("fragmento")` filtra por el texto de la consulta.

In [3]:
# Verdad de referencia y mapa consulta → texto, del caché compartido que usa el pipeline
dataset_cache_dir = cache.dataset_cache_base(
    CACHE_DIR, data.COUNTRY, data.DATASET_VERSION, data.MAX_WORD_COUNT
)
qrels, consulta_a_texto = data.get_pruned_qrels_and_queries(
    data.COUNTRY, data.DATASET_VERSION, kept_doc_ids=None,
    dataset_cache_dir=dataset_cache_dir, num_workers=1,
)
relevantes_por_consulta = qrels.to_dict()

print(f"Consultas de prueba: {len(consulta_a_texto):,}")
print(f"Verdad de referencia: {dataset_cache_dir / 'pruned_qrels.json'}")


def resolver_consulta(numero) -> str:
    """Devuelve el id de consulta (cadena) que corresponde al número dado.

    Se acepta como entero o como cadena; las claves de los runs y de los qrels son cadenas.
    """
    numero = str(numero)
    if numero not in consulta_a_texto:
        raise KeyError(
            f"La consulta {numero!r} no está en la partición de prueba. "
            "Usa listar_consultas() o buscar_consultas('...') para encontrar un número válido."
        )
    return numero


def listar_consultas(inicio: int = 0, cuantas: int = 10) -> None:
    """Imprime una rebanada de la partición, con el número y el texto de cada consulta."""
    for numero in list(consulta_a_texto)[inicio:inicio + cuantas]:
        print(f"{numero}  {consulta_a_texto[numero]}")


def buscar_consultas(fragmento: str, limite: int = 20) -> None:
    """Imprime las consultas cuyo texto contiene `fragmento` (sin distinguir mayúsculas)."""
    fragmento = fragmento.lower()
    encontradas = 0
    for numero, texto in consulta_a_texto.items():
        if fragmento in texto.lower():
            print(f"{numero}  {texto}")
            encontradas += 1
            if encontradas == limite:
                break
    print(f"\n{encontradas} consulta(s) mostradas (límite {limite}).")


# Ejemplos de uso para encontrar un número de consulta
listar_consultas(0, 5)
buscar_consultas("moneda circula en aruba")

15:17:59 | INFO | ⏱  START: Loading cached pruned qrels


15:17:59 | INFO | ⏱  DONE:  Loading cached pruned qrels (0.3s)


Consultas de prueba: 170,055
Verdad de referencia: /home/jmendoza/.cache/messirve_embeddings/shared_datasets/full_v1.2_nofilter/pruned_qrels.json
7397859   en grecia quién aplico la democracia radical
7397860   que conoces de la familia arduino
7397866  1 arroba cuantas kilogramos tiene
7397867  1 arroba cuantos kg tiene
7397868  1 arroba cuántas libras tiene
8101866  que moneda circula en aruba

1 consulta(s) mostradas (límite 20).


In [4]:
# Consulta configurada en la §1
CONSULTA = resolver_consulta(NUMERO_CONSULTA)
print(f"Consulta {CONSULTA}: {consulta_a_texto[CONSULTA]}")
for docid in relevantes_por_consulta[CONSULTA]:
    print(f"  relevante: {docid}")

Consulta 8101866: que moneda circula en aruba
  relevante: 328242#0


## 4. Top n por modelo

Se lee el run de cada modelo (ranx lo guarda como `{consulta: {docid: score}}`), se ordenan los documentos de esa consulta por score descendente —con orden estable, para respetar el orden del run cuando hay empates— y se toman los `n` primeros. Cada run se libera antes de cargar el siguiente, de modo que no están los seis rankings en memoria a la vez.

Leer un run cuesta unos segundos (≈150 MB comprimidos, ≈2.5 GB en memoria) y se repite en cada llamada: los seis modelos tardan ≈30 s, así que conviene tener claro qué consultas se quieren inspeccionar antes de ejecutar la celda varias veces.

In [5]:
def top_n_por_modelo(numero, n: int = N) -> dict[str, pd.DataFrame]:
    """Devuelve, por modelo, el top n de documentos recuperados para una consulta de prueba.

    Returns:
        {alias: DataFrame} con columnas: posicion, docid, puntaje, relevante.
    """
    numero = resolver_consulta(numero)
    relevantes = relevantes_por_consulta[numero]
    resultados: dict[str, pd.DataFrame] = {}

    for modelo in MODELOS:
        ruta = ruta_run(modelo)
        t0 = time.time()
        run = load_lz4(str(ruta))       # mismo dict que escribió ranx.Run.save
        if numero not in run:
            raise KeyError(f"El run de {modelo.alias} no contiene la consulta {numero}: {ruta}")

        top = sorted(run[numero].items(), key=lambda par: par[1], reverse=True)[:n]
        del run
        gc.collect()

        resultados[modelo.alias] = pd.DataFrame([
            {
                "posicion": posicion,
                "docid": docid,
                "puntaje": puntaje,
                "relevante": "sí" if docid in relevantes else "",
            }
            for posicion, (docid, puntaje) in enumerate(top, start=1)
        ])

        aviso = "" if len(top) == n else f"  (el run solo devolvió {len(top)})"
        print(f"{modelo.alias:>13}: {len(top):2d} documentos en {time.time() - t0:.1f} s{aviso}")

    return resultados

In [6]:
# Top n de la consulta configurada en la §1 (la que resolvió la §3). Para otra consulta, usa la
# «Versión interactiva» del final:  analizar_consulta(7521003, N)
resultados = top_n_por_modelo(CONSULTA, N)

         bm25: 10 documentos en 4.8 s


    splade-v3: 10 documentos en 4.7 s


     e5-large: 10 documentos en 4.7 s


       bge-m3: 10 documentos en 4.7 s


   qwen3-0.6b: 10 documentos en 4.7 s


jina-v5-small: 10 documentos en 4.7 s


## 5. Textos de los documentos

`cargar_textos(resultados, numero)` busca en el corpus de párrafos de la Wikipedia en español (`eswiki_20240401`) los `docid` que aparecen en las tablas —los recuperados y los relevantes— y devuelve su título y su texto. El corpus está en la caché de HuggingFace y se lee mapeado en disco: con una máscara de pyarrow se materializan solo los párrafos pedidos (≈40), no los 14 millones del corpus.

Los textos son **de una consulta**: al cambiar de consulta hay que volver a llamar a la función (lo hace `analizar_consulta` en la «Versión interactiva»). Si `INCLUIR_TEXTO = False`, no se carga nada y las tablas salen sin título ni extracto.

In [7]:
import datasets
import pyarrow as pa
import pyarrow.compute as pc


def cargar_textos(resultados, numero) -> dict[str, tuple[str, str]]:
    """Título y texto de los párrafos de una consulta: los recuperados y los relevantes.

    Returns:
        {docid: (título, texto)}; diccionario vacío si INCLUIR_TEXTO es False.
    """
    if not INCLUIR_TEXTO:
        print("INCLUIR_TEXTO = False: las tablas salen sin título ni extracto.")
        return {}

    docids = sorted(
        set().union(*[set(tabla["docid"]) for tabla in resultados.values()])
        | set(relevantes_por_consulta[numero])
    )

    t0 = time.time()
    corpus = datasets.load_dataset(data.CORPUS_NAME, split="corpus")
    tabla_corpus = corpus.data.table
    columna_docid = pc.cast(tabla_corpus["docid"], pa.string())
    seleccion = tabla_corpus.filter(
        pc.is_in(columna_docid, value_set=pa.array(docids, type=columna_docid.type))
    )

    textos: dict[str, tuple[str, str]] = {}
    for docid, titulo, texto in zip(
        seleccion["docid"].to_pylist(),
        seleccion["title"].to_pylist(),
        seleccion["text"].to_pylist(),
    ):
        textos[str(docid)] = (titulo or "", texto or "")

    print(f"Párrafos pedidos: {len(docids)} | encontrados en el corpus: {len(textos)} "
          f"({time.time() - t0:.1f} s)")
    return textos


# Textos de la consulta configurada en la §1. Son de una consulta: al cambiar de consulta hay
# que volver a llamar a la función (lo hace `analizar_consulta` en la «Versión interactiva»).
textos = cargar_textos(resultados, CONSULTA)

Párrafos pedidos: 38 | encontrados en el corpus: 38 (1.6 s)


## 6. Resultados

El top n de cada modelo para la consulta configurada. `posicion` es el lugar que el modelo le da al documento (1 = primero), `puntaje` es el score del run y `relevante` marca los documentos que la verdad de referencia considera relevantes para esta consulta.

`mostrar_resultados(resultados, numero, textos)` y `mostrar_relevantes(numero, textos)` reciben la consulta, así que sirven para cualquier número (es lo que usa la «Versión interactiva»). Después de las tablas se imprime el **texto completo de los documentos relevantes**, para poder juzgar lo que cada modelo devolvió.

In [8]:
def _extracto(texto: str, limite: int = 330) -> str:
    """Primeras palabras del documento, para que la tabla se pueda leer."""
    texto = " ".join(texto.split())
    if len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + " […]"


def mostrar_resultados(resultados, numero, textos) -> None:
    """Muestra el top n de cada modelo, con título y extracto si hay textos cargados.

    Recibe la consulta (`numero`) y sus `textos` en vez de leerlos de variables globales: así
    sirve para cualquier consulta, no solo para la configurada en la §1.
    """
    for alias, tabla in resultados.items():
        vista = tabla.copy()
        if textos:
            vista["titulo"] = [textos.get(docid, ("", ""))[0] for docid in vista["docid"]]
            vista["extracto"] = [_extracto(textos.get(docid, ("", ""))[1]) for docid in vista["docid"]]
        display(Markdown(f"**{alias}** — top {len(vista)} de «{consulta_a_texto[numero]}»"))
        display(vista)


def mostrar_relevantes(numero, textos) -> None:
    """Imprime el texto completo de los documentos relevantes de una consulta."""
    display(Markdown("**Documentos relevantes (verdad de referencia)**"))
    for docid in relevantes_por_consulta[numero]:
        titulo, texto = textos.get(docid, ("(sin texto)", ""))
        print(f"{docid} — {titulo}\n{texto}\n")


# Resultados de la consulta configurada en la §1
mostrar_resultados(resultados, CONSULTA, textos)
mostrar_relevantes(CONSULTA, textos)

**bm25** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,10403#24,10.3992,,Aruba,"Aruba es parte del Reino de los Países Bajos, ..."
1,2,15756#0,10.3215,,Gobierno y política de Aruba,"Aruba es parte del Reino de los Países Bajos, ..."
2,3,943022#4,9.7543,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y ..."
3,4,1992977#7,9.6735,,Unión monetaria de América del Norte,Existirían antecedentes de esa Unión. Se han p...
4,5,5585611#0,9.6443,,Museo numismático de Aruba,El Museo numismático de Aruba es un museo esta...
5,6,8895256#8,9.0573,,Paradoja de las ruedas de Aristóteles,"El autor de ""Falacias y paradojas matemáticas""..."
6,7,35612#0,8.9010,,Florín neerlandés,El florín neerlandés () (conocido erróneamente...
7,8,7724302#5,8.8870,,Paradoja de la moneda que gira,Otra forma intuitiva de entender este problema...
8,9,3879186#0,8.8491,,Franco neohebridense,El franco fue la moneda del condominio anglo-f...
9,10,328242#0,8.8355,sí,Florín arubeño,El florín es la moneda oficial de Aruba. Se di...


**splade-v3** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,1992977#7,18.231495,,Unión monetaria de América del Norte,Existirían antecedentes de esa Unión. Se han p...
1,2,2476004#45,17.676958,,Moneda (divisa),"Cuando la letra circula, el que circula es sim..."
2,3,1096476#3,17.478882,,Dinar argelino,Las monedas que circulan con mayor asiduidad s...
3,4,10403#135,16.608856,,Aruba,"El 19 de febrero de 2013, Arubus puso en march..."
4,5,328242#0,16.016756,sí,Florín arubeño,El florín es la moneda oficial de Aruba. Se di...
5,6,2472607#19,15.500867,,Monedas del reino visigodo,Se han contabilizado casi 100 talleres de fabr...
6,7,616947#16,15.496761,,Quetzal (moneda),El billete de cincuenta centavos de quetzal se...
7,8,164347#14,15.393028,,Modúbar de la Emparedada,La iglesia de San Cristóbal de Cojóbar data su...
8,9,202538#1,15.314874,,Lempira (moneda),Circulan monedas de 5 y 10 centavos (aleación ...
9,10,2476004#60,15.311818,,Moneda (divisa),Esta nueva forma de circulación monetaria es l...


**e5-large** — top 10 de «que moneda circula en aruba»

...

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,328242#0,0.778951,sí,Florín arubeño,El florín es la moneda oficial de Aruba. Se di...
1,2,7733876#2,0.733029,,Banco Central de Aruba,"El banco es una entidad legal en sí misma (""su..."
2,3,10403#54,0.724236,,Aruba,Alrededor del 70 % del PIB de Aruba proviene d...
3,4,3655473#0,0.696541,,Economía de Aruba,La Economía de Aruba es de libre mercado. El t...
4,5,7733876#0,0.670403,,Banco Central de Aruba,El Banco Central de Aruba () es el banco centr...
5,6,943022#4,0.669887,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y ..."
6,7,10403#56,0.658956,,Aruba,Aruba es un país próspero. El desempleo es baj...
7,8,3655473#11,0.655498,,Economía de Aruba,gastos: 577.9 millones de dólares (2005 estimado)
8,9,3655935#7,0.650602,,Idiomas de Aruba,Aruba tiene 4 periódicos publicados en Papiame...
9,10,9567892#43,0.644124,,Pandemia de COVID-19 en Aruba,1 de mayo de 2020: el gobierno holandés aprobó...


**jina-v5-small** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,328242#0,0.817917,sí,Florín arubeño,El florín es la moneda oficial de Aruba. Se di...
1,2,7733876#2,0.788228,,Banco Central de Aruba,"El banco es una entidad legal en sí misma (""su..."
2,3,328242#1,0.712161,,Florín arubeño,"Las monedas tienen denominaciones de 1, 2, 5 y..."
3,4,10403#54,0.707369,,Aruba,Alrededor del 70 % del PIB de Aruba proviene d...
4,5,943022#4,0.706751,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y ..."
5,6,5585611#0,0.698622,,Museo numismático de Aruba,El Museo numismático de Aruba es un museo esta...
6,7,7733876#0,0.675414,,Banco Central de Aruba,El Banco Central de Aruba () es el banco centr...
7,8,328242#2,0.668061,,Florín arubeño,La moneda tiene una tasa de cambio con el dóla...
8,9,68602#73,0.664497,,Bonaire,"En 2011, las Islas BES sustituyeron su moneda,..."
9,10,35612#0,0.663878,,Florín neerlandés,El florín neerlandés () (conocido erróneamente...


**Documentos relevantes (verdad de referencia)**

328242#0 — Florín arubeño
El florín es la moneda oficial de Aruba. Se divide en 100 céntimos. Fue introducido en 1986, reemplazando a paridad el Florín antillano neerlandés.



## 7. Dónde aparece cada documento relevante

Para cada documento relevante de la consulta, la posición que le asigna cada modelo dentro de su top n (`—` si no aparece en él). La última columna cuenta en cuántos de los seis modelos aparece.

In [9]:
def resumen_relevantes(resultados, numero) -> pd.DataFrame:
    """Posición de cada documento relevante en el top n de cada modelo ('—' si no aparece)."""
    filas = []
    for docid in relevantes_por_consulta[numero]:
        fila = {"docid": docid}
        for alias, tabla in resultados.items():
            posiciones = tabla.index[tabla["docid"] == docid]
            fila[alias] = int(tabla.loc[posiciones[0], "posicion"]) if len(posiciones) else "—"
        fila["modelos"] = sum(1 for alias in resultados if fila[alias] != "—")
        filas.append(fila)
    return pd.DataFrame(filas)


display(resumen_relevantes(resultados, CONSULTA))

,docid,bm25,splade-v3,e5-large,bge-m3,qwen3-0.6b,jina-v5-small,modelos
0,328242#0,10,5,1,1,1,1,6


## Versión interactiva

Las secciones anteriores están atadas a la consulta de la §1 y dejan un ejemplo ejecutado en el cuaderno. Para inspeccionar **otras consultas** está `analizar_consulta(numero, n)`: recupera el top n de los seis modelos, carga los textos de esa consulta y muestra el resumen y las tablas. Tarda ≈30 s (leer los seis runs) y no hace falta volver a ejecutar nada de arriba.

`buscar_consultas` sirve para encontrar un número; en la última celda se cambia el número y se vuelve a ejecutar solo esa celda.

In [10]:
buscar_consultas("presidente")

7400945  a cuál batalla asistió el presidente mora
7423982  a quien propuso el rey como presidente
7430125  a qué presidente mexicano se le atribuye el apoyo para la apertura del primer periódico
7483566  como vuela el presidente de estados unidos
7494312  con que presidente inicia el neoliberalismo en mexico
7494314  con que presidente se dio el auge petrolero
7494315  con que presidente se inicio el neoliberalismo en mexico
7510540  cual es el presidente ruso
7512242  cual es el vicepresidente del ecuador
7516770  cual es la residencia oficial del presidente de la república dominicana
7520638  cual fue el presidente que duro menos tiempo en la presidencia
7520639  cual fue el presidente que duro menos tiempo en la presidencia argentina
7520753  cual fue el primer presidente de rd
7520878  cual fue el segundo presidente de estados unidos
7520880  cual fue el segundo presidente de usa
7521003  cual fue el ultimo presidente de la urss
7521004  cual fue el ultimo presidente militar
75217

In [11]:
def analizar_consulta(numero, n: int | None = None) -> dict[str, pd.DataFrame]:
    """Recupera y muestra el top n de una consulta: la vía interactiva del cuaderno.

    Hace el recorrido completo para el número que reciba —top n por modelo, textos de esa
    consulta, resumen de los relevantes y tablas—, así que no depende de las variables de las
    secciones anteriores.

    Args:
        numero: número (id) de la consulta de prueba; ver `buscar_consultas`.
        n: cuántos documentos por modelo; por omisión, el `N` configurado en la §1.

    Returns:
        {alias: DataFrame}, los mismos resultados que muestra.
    """
    n = N if n is None else n
    numero = resolver_consulta(numero)
    resultados = top_n_por_modelo(numero, n)
    textos = cargar_textos(resultados, numero)

    display(Markdown(f"### Consulta {numero}: «{consulta_a_texto[numero]}»"))
    mostrar_relevantes(numero, textos)
    display(resumen_relevantes(resultados, numero))
    mostrar_resultados(resultados, numero, textos)
    return resultados


# Cambia el número (y, si quieres, n) y vuelve a ejecutar solo esta celda.
NUMERO_CONSULTA = 7521003
N = 10

resultados = analizar_consulta(NUMERO_CONSULTA, N)

         bm25: 10 documentos en 4.7 s


    splade-v3: 10 documentos en 4.8 s


     e5-large: 10 documentos en 4.7 s


       bge-m3: 10 documentos en 4.7 s


   qwen3-0.6b: 10 documentos en 4.7 s


...

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,4903115#7,0.557072,,Disolución de la Unión Soviética,El 15 de marzo de 1990 Gorbachov fue elegido p...
1,2,7955585#3,0.556943,,Partido de la Unión Republicana Socialista,El Partido de la Unión Republicana Socialista ...
2,3,10362682#7,0.555931,,Younoussi Touré,En el III Congreso Ordinario de la URD en novi...
3,4,2539671#2,0.551864,,Unión Social Republicana de Asalariados de Chile,José Santos Salas fue su primer y único candid...
4,5,10181232#41,0.548934,,Historia contemporánea de Ucrania,"El 25 de enero de 1992, Mijaíl Gorbachov renun..."
5,6,6885018#0,0.546886,,Luis Usera,"Luis Usera y Bugallal (Talavera de la Reina, 8..."
6,7,9775313#1,0.546139,,Ruslán Jasbulátov,Ruslán Jasbulátov fue el último en ocupar el c...
7,8,952701#48,0.544932,,Historia de la Unión Soviética (1985-1991),"El 25 de diciembre de 1991 Gorbachov, cediendo..."
8,9,303630#0,0.544490,sí,Presidente de la Unión Soviética,"El presidente de la Unión Soviética (del : ""Пр..."
9,10,9988634#6,0.533200,,Krishna Urs,Urs fue confirmado por el Senado de Estados Un...


**qwen3-0.6b** — top 10 de «cual fue el ultimo presidente de la urss»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,303630#0,0.669163,sí,Presidente de la Unión Soviética,"El presidente de la Unión Soviética (del : ""Пр..."
1,2,952701#48,0.664754,,Historia de la Unión Soviética (1985-1991),"El 25 de diciembre de 1991 Gorbachov, cediendo..."
2,3,2432532#2,0.634633,,Historia de la Unión Soviética (1953-1985),"Después de la muerte de Iósif Stalin, el 5 de ..."
3,4,303630#8,0.628020,,Presidente de la Unión Soviética,"El 12 de febrero de 1990, en el Kremlin, bajo ..."
4,5,2922544#2,0.626941,,Presidente de Ucrania,Desde el establecimiento del cargo el 5 de jul...
5,6,225982#3,0.622941,,Presidente de Rusia,El actual presidente de la Federación de Rusia...
6,7,714592#11,0.612952,,Vasili Kuznetsov (político),"A principios de la década de 1980, fue preside..."
7,8,3338021#1,0.610992,,Presidente de Uzbekistán,Islam Karimov fue el único presidente de Uzbek...
8,9,303630#1,0.610506,,Presidente de la Unión Soviética,"Desde la década de 1970, la mayoría de los pod..."
9,10,2162786#3,0.609434,,Anexo: Gobernantes de la Unión Soviética,Vladímir Lenin fue elegido como Presidente del...


**jina-v5-small** — top 10 de «cual fue el ultimo presidente de la urss»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,303630#0,0.737167,sí,Presidente de la Unión Soviética,"El presidente de la Unión Soviética (del : ""Пр..."
1,2,9775313#1,0.708947,,Ruslán Jasbulátov,Ruslán Jasbulátov fue el último en ocupar el c...
2,3,303630#1,0.700318,,Presidente de la Unión Soviética,"Desde la década de 1970, la mayoría de los pod..."
3,4,952701#48,0.691883,,Historia de la Unión Soviética (1985-1991),"El 25 de diciembre de 1991 Gorbachov, cediendo..."
4,5,35247#0,0.658380,,Borís Yeltsin,"Borís Nikoláievich Yeltsin (, ; Butká, óblast ..."
5,6,226000#33,0.644297,,Política de Rusia,"Gorbachov, tras conocer los detalles del Trata..."
6,7,714592#11,0.643060,,Vasili Kuznetsov (político),"A principios de la década de 1980, fue preside..."
7,8,303630#10,0.641901,,Presidente de la Unión Soviética,"El 14 de marzo de 1990, el congreso eligió a M..."
8,9,915091#35,0.640840,,Yuri Andrópov,"A los 69 años de edad, la enfermedad renal que..."
9,10,7802835#2,0.640290,,Víktor Geráshchenko,En 1982 Geráshchenko pasó a trabajar en el Vne...
